# 🛡️ MVP — Detecção de Anomalias em Segurança Cibernética

Este notebook implementa um MVP para identificar comportamentos potencialmente anômalos em registros de acesso utilizando **Isolation Forest**, um algoritmo de aprendizado de máquina não supervisionado.

**Importante:** os dados utilizados neste MVP são simulados para fins acadêmicos e de prototipagem. Uma anomalia identificada não representa, por si só, a confirmação de um ataque.

## 1. Objetivos

- Gerar ou carregar registros de acesso;
- realizar pré-processamento;
- explorar os dados;
- treinar o modelo Isolation Forest;
- identificar registros potencialmente anômalos;
- calcular um indicador auxiliar de risco;
- visualizar os resultados;
- exportar a base analisada.

In [ ]:
# Instalação das bibliotecas — útil no Google Colab.
# Se estiver executando localmente e as bibliotecas já estiverem instaladas,
# esta célula pode ser executada normalmente ou ignorada.

!pip -q install pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

from IPython.display import display

SEED = 42
np.random.seed(SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('Bibliotecas carregadas com sucesso.')

## 2. Configurações do MVP

Os parâmetros abaixo podem ser alterados para testar diferentes cenários.

In [ ]:
N_REGISTROS = 1500
CONTAMINATION = 0.05  # proporção esperada de anomalias
OUTPUT_DIR = 'results'
DATA_DIR = 'data'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print(f'Registros planejados: {N_REGISTROS}')
print(f'Contamination do modelo: {CONTAMINATION:.0%}')

## 3. Geração do conjunto de dados simulado

O dataset representa eventos de acesso e contém características como tentativas de login, duração da sessão, volume de dados, falhas de autenticação, portas utilizadas e acesso externo.

Uma pequena parcela dos registros recebe padrões extremos para representar situações potencialmente anômalas.

In [ ]:
rng = np.random.default_rng(SEED)

n_anomalias_simuladas = max(1, int(N_REGISTROS * 0.05))
n_normais = N_REGISTROS - n_anomalias_simuladas

timestamps = pd.date_range(
    start='2026-01-01',
    periods=N_REGISTROS,
    freq='min'
)
rng.shuffle(timestamps.values)

normal = pd.DataFrame({
    'timestamp': timestamps[:n_normais],
    'login_attempts': rng.poisson(2, n_normais) + 1,
    'session_duration': np.clip(rng.gamma(2.5, 18, n_normais), 2, 180),
    'bytes_sent': np.clip(rng.lognormal(8.0, 0.75, n_normais), 500, 150000),
    'bytes_received': np.clip(rng.lognormal(9.0, 0.85, n_normais), 1000, 300000),
    'failed_logins': rng.poisson(0.4, n_normais),
    'unique_ports': np.clip(rng.poisson(2, n_normais) + 1, 1, 10),
    'is_external_ip': rng.binomial(1, 0.18, n_normais)
})

anom = pd.DataFrame({
    'timestamp': timestamps[n_normais:],
    'login_attempts': rng.integers(15, 80, n_anomalias_simuladas),
    'session_duration': rng.uniform(1, 8, n_anomalias_simuladas),
    'bytes_sent': rng.uniform(200000, 1500000, n_anomalias_simuladas),
    'bytes_received': rng.uniform(300000, 2500000, n_anomalias_simuladas),
    'failed_logins': rng.integers(8, 40, n_anomalias_simuladas),
    'unique_ports': rng.integers(12, 60, n_anomalias_simuladas),
    'is_external_ip': rng.binomial(1, 0.85, n_anomalias_simuladas)
})

df = pd.concat([normal, anom], ignore_index=True)
df['access_hour'] = df['timestamp'].dt.hour
df['ip_address'] = [f'192.168.{rng.integers(0, 255)}.{rng.integers(1, 255)}' for _ in range(len(df))]
df['synthetic_label'] = np.r_[np.zeros(n_normais, dtype=int), np.ones(n_anomalias_simuladas, dtype=int)]

df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv(os.path.join(DATA_DIR, 'acessos_ciberneticos.csv'), index=False)

print(f'Dataset criado: {len(df):,} registros')
print(f'Anomalias simuladas para validação: {df["synthetic_label"].sum():,}')
            

## 4. Visualização inicial dos dados

In [ ]:
display(df.head(10))

print('\nInformações do dataset:')
print(df.info())

print('\nValores ausentes por coluna:')
display(df.isna().sum().to_frame('valores_ausentes'))

## 5. Análise exploratória

A etapa exploratória ajuda a entender a distribuição das variáveis antes da aplicação do modelo.

In [ ]:
numeric_cols = [
    'login_attempts', 'session_duration', 'bytes_sent',
    'bytes_received', 'failed_logins', 'unique_ports',
    'access_hour', 'is_external_ip'
]

display(df[numeric_cols].describe().T)

In [ ]:
plt.figure(figsize=(10, 5))
for label, group in df.groupby('synthetic_label'):
    nome = 'Anomalia simulada' if label == 1 else 'Normal'
    plt.scatter(group['failed_logins'], group['unique_ports'], alpha=0.45, s=18, label=nome)
plt.title('Falhas de login × Portas utilizadas')
plt.xlabel('Falhas de login')
plt.ylabel('Quantidade de portas')
plt.legend()
plt.show()


## 6. Preparação das variáveis

O modelo utiliza apenas variáveis numéricas relacionadas ao comportamento do acesso. O endereço IP não é usado diretamente no treinamento.

In [ ]:
features = [
    'login_attempts',
    'session_duration',
    'bytes_sent',
    'bytes_received',
    'failed_logins',
    'unique_ports',
    'access_hour',
    'is_external_ip'
]

X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Variáveis utilizadas:')
for feature in features:
    print('-', feature)

## 7. Treinamento do Isolation Forest

In [ ]:
model = IsolationForest(
    n_estimators=200,
    contamination=CONTAMINATION,
    random_state=SEED,
    n_jobs=-1
)

model.fit(X_scaled)

# O Isolation Forest retorna 1 para observações consideradas normais
# e -1 para observações consideradas anômalas.
df['model_prediction'] = model.predict(X_scaled)
df['anomaly_score'] = -model.score_samples(X_scaled)
df['anomaly_flag'] = (df['model_prediction'] == -1).astype(int)
df['classification'] = np.where(df['anomaly_flag'] == 1, 'Anomalia', 'Normal')

print('Modelo treinado com sucesso.')

## 8. Indicador auxiliar de risco

Além da classificação do modelo, é calculado um indicador auxiliar de 0 a 100. Ele não representa uma probabilidade estatística de ataque; serve apenas para facilitar a priorização dos registros dentro do MVP.

In [ ]:
score_min = df['anomaly_score'].min()
score_max = df['anomaly_score'].max()

if score_max == score_min:
    df['risk_score'] = 0.0
else:
    df['risk_score'] = 100 * (df['anomaly_score'] - score_min) / (score_max - score_min)

df['risk_level'] = pd.cut(
    df['risk_score'],
    bins=[-np.inf, 25, 50, 75, np.inf],
    labels=['Baixo', 'Moderado', 'Alto', 'Crítico']
)

display(df[['anomaly_score', 'risk_score', 'risk_level', 'classification']].head(10))

## 9. Indicadores do MVP

In [ ]:
total = len(df)
anomalias = int(df['anomaly_flag'].sum())
normais = total - anomalias
percentual_anomalias = 100 * anomalias / total

print(f'Total de registros analisados: {total:,}')
print(f'Registros normais: {normais:,}')
print(f'Anomalias identificadas: {anomalias:,}')
print(f'Percentual de anomalias: {percentual_anomalias:.2f}%')

In [ ]:
plt.figure(figsize=(10, 5))
for label, group in df.groupby('classification'):
    plt.hist(group['risk_score'], bins=25, alpha=0.6, label=label)
plt.title('Distribuição do indicador auxiliar de risco')
plt.xlabel('Risk Score (0–100)')
plt.ylabel('Quantidade')
plt.legend()
plt.show()


## 10. Registros potencialmente anômalos

A tabela abaixo apresenta os eventos classificados pelo modelo como anômalos, ordenados pelo indicador auxiliar de risco.

In [ ]:
anomalias_df = (
    df[df['anomaly_flag'] == 1]
    .sort_values('risk_score', ascending=False)
    .copy()
)

colunas_exibicao = [
    'timestamp', 'ip_address', 'login_attempts', 'failed_logins',
    'unique_ports', 'is_external_ip', 'anomaly_score',
    'risk_score', 'risk_level', 'classification'
]

display(anomalias_df[colunas_exibicao].head(20))

## 11. Comparação com os rótulos simulados

Como este MVP utiliza dados sintéticos, existe um rótulo artificial (`synthetic_label`) apenas para avaliar o comportamento do protótipo. Em uma aplicação real, esse rótulo não estaria necessariamente disponível.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

y_true = df['synthetic_label']
y_pred = df['anomaly_flag']

print('Métricas sobre os dados sintéticos:')
print(f'Acurácia : {accuracy_score(y_true, y_pred):.3f}')
print(f'Precisão : {precision_score(y_true, y_pred, zero_division=0):.3f}')
print(f'Recall   : {recall_score(y_true, y_pred, zero_division=0):.3f}')
print(f'F1-score : {f1_score(y_true, y_pred, zero_division=0):.3f}')

print('\nMatriz de confusão:')
display(pd.DataFrame(
    confusion_matrix(y_true, y_pred),
    index=['Real normal', 'Real anomalia'],
    columns=['Predito normal', 'Predito anomalia']
))

## 12. Exportação dos resultados

In [ ]:
resultado_path = os.path.join(OUTPUT_DIR, 'resultados_anomalias.csv')
df.to_csv(resultado_path, index=False)

anomalias_path = os.path.join(OUTPUT_DIR, 'anomalias_identificadas.csv')
anomalias_df.to_csv(anomalias_path, index=False)

print('Arquivos exportados:')
print('-', resultado_path)
print('-', anomalias_path)

## 13. Conclusão do MVP

O protótipo demonstra um fluxo básico de detecção de anomalias em registros de acesso. O Isolation Forest permite identificar observações que apresentam comportamento diferente do conjunto analisado, enquanto o indicador auxiliar de risco organiza os resultados para facilitar uma eventual investigação.

Para utilização em um ambiente real, seria necessário validar o modelo com dados reais e representativos, estabelecer critérios operacionais, tratar questões de privacidade e segurança, monitorar falsos positivos e falsos negativos e integrar a solução ao ambiente de monitoramento existente.

## 🚀 Próximos passos

1. Substituir os dados simulados por logs reais e anonimizados.
2. Avaliar diferentes parâmetros do Isolation Forest.
3. Comparar o modelo com outras técnicas de detecção de anomalias.
4. Criar um dashboard interativo.
5. Implementar alertas para eventos de maior prioridade.
6. Integrar o MVP a uma arquitetura de monitoramento/SIEM.
7. Avaliar o desempenho continuamente com dados históricos.